# 🐳 DOCKER — COMPLETE REVISION NOTES
### Phases 1–7 · Modules 1–23 · AI/ML deployment focus

> **Purpose:** Fast, structured revision of the complete Docker syllabus. These notes combine the concepts, commands, comparisons, workflows, and AI/ML examples covered across all seven phases.

---

## How to revise this notebook

1. Read the **mental model** for each phase.
2. Explain each concept aloud without looking.
3. Reproduce the important commands from memory.
4. Use the comparison tables to distinguish similar concepts.
5. Finish with the **end-to-end deployment workflow** and self-test.

**Big picture:** Docker packages an application and its runtime environment into images, runs those images as containers, connects and persists services, and helps deploy and operate them consistently.

# 🧭 MASTER ROADMAP — ALL 7 PHASES

| Phase | Modules | Main focus |
|---|---:|---|
| 1. Understanding applications | 1–3 | Application components, environments, architecture |
| 2. Docker fundamentals | 4–7 | Docker, architecture, images, containers |
| 3. Building & managing containers | 8–10 | Dockerfile, lifecycle, ports |
| 4. Storage, networking & configuration | 11–13 | Volumes, networks, environment variables |
| 5. Docker Compose | 14–16 | Multi-container applications |
| 6. Production Docker & deployment | 17–20 | Debugging, registries, deployment, scaling |
| 7. Security, optimization & project | 21–23 | Secure images, efficient builds, AI/ML deployment |

## The complete mental model

```text
APPLICATION
  ├── Frontend / client
  ├── Backend API
  ├── AI model service
  └── Database / cache
          │
          ▼
DEPENDENCIES + RUNTIME + CONFIGURATION
          │
          ▼
       DOCKERFILE
          │ docker build
          ▼
      DOCKER IMAGE
          │ docker run / Compose
          ▼
       CONTAINER(S)
          ├── Ports → access
          ├── Volumes → persistent data
          ├── Networks → communication
          └── Environment → runtime config
                  │
                  ▼
       REGISTRY → SERVER → DEPLOYMENT
                  │
                  ▼
       Logs / health / security / scaling
```

# 🧠 PHASE 1 — UNDERSTANDING APPLICATIONS

## Module 1 — How applications work

### Application components

| Component | Meaning |
|---|---|
| Frontend | User-facing interface |
| Backend | Server-side application logic |
| Server | Computer/process that provides a service |
| API | Contract/interface through which software components communicate |
| HTTP | Request-response protocol commonly used by web APIs |
| Database | Persistent storage for application data |
| AI inference service | Loads a trained model and produces predictions |

### Typical request flow

```text
User → Frontend → Backend API → (Model service / Database)
     ←           ←             ←
                 HTTP response
```

- **Request:** client sends method, URL, headers and possibly a body.
- **Response:** server returns a status code, headers and often a body.
- **Inference:** using a trained model to produce a prediction.
- Separating model inference from the backend can allow independent deployment, scaling and resource management.

## Module 2 — Software environments & dependency problems

- **Environment:** runtime context in which software executes.
- **Runtime:** software required to execute a program, e.g. Python.
- **Dependency:** external package/library required by an application.
- **Dependency conflict:** incompatible package or version requirements.
- **Virtual environment:** isolates Python packages for a project.
- **Version pinning:** specifies dependency versions to improve reproducibility.
- **Environment variables:** runtime key-value configuration.
- **“Works on my machine”:** differences in OS, runtime, packages, versions, configuration or files cause inconsistent behavior.

### Python virtual environment commands

```bash
python -m venv .venv

# Windows PowerShell
.venv\Scripts\Activate.ps1

# macOS / Linux
source .venv/bin/activate

python -m pip install -r requirements.txt
python -m pip freeze > requirements.txt
```

**What virtual environments solve:** Python package isolation.

**What they do not fully solve:** OS-level differences, system libraries, runtime differences, deployment configuration and all machine-level dependencies. Docker helps package more of the application environment.

## Module 3 — Application architecture

### Monolith vs multi-service

| Monolithic | Multi-service |
|---|---|
| Components packaged/deployed together | Components separated into services |
| Simpler initial deployment | Independent deployment possible |
| Components often scale together | Services can scale independently |
| Internal function calls | Network/API communication between services |

Example multi-service architecture:

```text
Frontend → Backend API → Model service
                     └→ Database
                     └→ Cache
```

- **Synchronous communication:** caller waits for a response.
- **Asynchronous communication:** caller and processing can be decoupled, often using queues/events.
- **Service dependency:** one service requires another service to function.
- **Architecture:** how application components are organized.
- **Infrastructure:** machines, networks, storage and runtime resources used to run the architecture.

### Phase 1 key takeaway

Docker is useful because applications consist of components with runtimes, dependencies, configuration and infrastructure needs. Docker helps package and run those components consistently.

# 🐳 PHASE 2 — DOCKER FUNDAMENTALS

## Module 4 — What is Docker?

**Docker** is a platform for building, distributing and running applications in containers.

It helps package application code and dependencies into an image, then run that image as an isolated container.

### What Docker addresses

- Inconsistent environments
- Dependency differences
- Repetitive manual setup
- Difficult application handoff between machines
- Deployment inconsistency

## Docker vs virtual machines

| Containers | Virtual machines |
|---|---|
| Share host OS kernel | Each VM runs a guest OS |
| Container isolation | Hardware virtualization |
| Usually lighter-weight | Usually more overhead |
| Start quickly in many workloads | Typically more boot overhead |
| Package app and dependencies | Package a full guest OS environment |

Containers are isolated processes, not miniature full virtual machines.

## Module 5 — Docker architecture

```text
User
 │ docker CLI command
 ▼
Docker Client
 │ Docker API
 ▼
Docker Daemon (dockerd)
 ├── Images
 ├── Containers
 ├── Networks
 └── Volumes
```

- **Docker CLI/client:** accepts commands such as `docker run`.
- **Docker daemon (`dockerd`):** manages Docker objects.
- **Docker API:** communication interface between client and daemon.
- **Docker Desktop:** desktop environment that provides Docker tooling and a Linux environment for containers on Windows/macOS.

### What happens during `docker run`?

1. CLI receives command.
2. Client communicates with daemon.
3. Daemon checks whether image exists locally.
4. If needed, image is pulled from a registry.
5. Container is created.
6. Container process is started.

## Module 6 — Docker images

**Image:** read-only template used to create containers.

**Container:** runnable instance of an image, with a writable container layer.

```text
Image → create/run → Container
```

### Important image concepts

- **Base image:** starting image, e.g. `python:3.12-slim`.
- **Layer:** filesystem/build step contributing to an image.
- **Image immutability:** image layers are treated as read-only; runtime writes go to the container writable layer.
- **Tag:** human-readable image reference such as `v1`, `3.12-slim`.
- `latest` is a tag, not a guarantee that an image is newest or production-safe.
- Images can be pulled from registries or built from Dockerfiles.

Commands:

```bash
docker images
docker pull python:3.12-slim
docker image inspect python:3.12-slim
docker history python:3.12-slim
```

## Module 7 — Containers

A container is an isolated running (or stopped) instance created from an image.

- Containers use isolation mechanisms such as namespaces and cgroups.
- A container has a writable layer above image layers.
- Removing a container removes its writable layer.
- Important data should be stored in volumes or other persistent storage.

### One image, multiple containers

```text
                 Image
              /    |    \
             ▼     ▼     ▼
        Container A B   C
```

### Multiple replicas vs multiple services

- **Replicas:** multiple instances of the same service.
- **Different services:** different application components, e.g. API, Redis and database.

### Image vs container

| Image | Container |
|---|---|
| Template/package | Instance |
| Read-only image layers | Image layers + writable layer |
| Can create many containers | Created from an image |
| Stored locally or in registry | Has runtime state |

### Phase 2 key takeaway

Docker architecture manages images, containers, networks and volumes. An image is the package; a container is the running instance.

# 🛠️ PHASE 3 — BUILDING & MANAGING CONTAINERS

## Module 8 — Dockerfile

A **Dockerfile** is a text file containing instructions to build an image.

### Core Dockerfile instructions

| Instruction | Purpose |
|---|---|
| `FROM` | Select base image |
| `WORKDIR` | Set working directory |
| `COPY` | Copy files from build context |
| `RUN` | Execute command during image build |
| `ENV` | Set image environment variable |
| `EXPOSE` | Document intended container port |
| `CMD` | Default command when container starts |
| `ENTRYPOINT` | Configure the main executable |

### FastAPI example

`main.py`:

```python
from fastapi import FastAPI

app = FastAPI()

@app.get("/")
def home():
    return {"message": "API is running"}

@app.get("/health")
def health():
    return {"status": "healthy"}
```

`requirements.txt`:

```text
fastapi
uvicorn
```

`Dockerfile`:

```dockerfile
FROM python:3.12-slim

WORKDIR /app

COPY requirements.txt .

RUN pip install --no-cache-dir -r requirements.txt

COPY . .

EXPOSE 8000

CMD ["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8000"]
```

### Build image

```bash
docker build -t sentiment-api:v1 .
```

- `-t`: tag/name the image.
- `.`: build context (current directory).
- Build context: files made available to the builder.
- `.dockerignore`: excludes unnecessary files from the build context.

### Build-time vs runtime

| Build time | Runtime |
|---|---|
| `FROM`, `COPY`, `RUN` | `CMD`, container process |
| Produces image | Starts container from image |
| Build context and cache matter | Ports, runtime env and mounts matter |

### Build cache principle

Docker can reuse cached build steps when their inputs have not changed. Put dependency manifests before frequently changing source files:

```dockerfile
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt
COPY . .
```

This helps avoid reinstalling dependencies for every source-code change.

## Module 9 — Container lifecycle

```text
docker create → created container
docker start  → start existing container
docker stop   → stop running container
docker restart→ restart container
docker rm     → remove stopped container
```

`docker run` combines creation and startup.

### Essential commands

```bash
docker run -d --name my-app nginx
docker create --name my-app nginx
docker start my-app
docker stop my-app
docker restart my-app
docker rm my-app

docker ps
docker ps -a

docker exec -it my-app sh
```

- `-d`: detached/background mode.
- `-it`: interactive terminal.
- `--name`: assign a container name.
- `exec`: run a command inside an already-running container.
- `run`: create a new container and start its main process.

## Module 10 — Ports & port mapping

A port identifies a network endpoint used by an application.

```bash
docker run -d -p 8080:8000 sentiment-api:v1
```

Mapping format:

```text
HOST_PORT : CONTAINER_PORT
8080      : 8000
```

Request path:

```text
Browser → localhost:8080 → Docker forwarding → container:8000
```

- **Host port:** port exposed on the host machine.
- **Container port:** port where the application listens inside the container.
- `EXPOSE 8000` is image metadata/documentation; it does not publish a port.
- A service inside a container commonly needs to bind to `0.0.0.0` to accept traffic through published ports.
- `127.0.0.1` inside the container refers to that container's loopback interface.

Local-only host binding:

```bash
docker run -p 127.0.0.1:8080:8000 sentiment-api:v1
```

### Phase 3 key takeaway

Dockerfile builds an image; lifecycle commands manage containers; port publishing connects host traffic to a container service.

# 💾 PHASE 4 — STORAGE, NETWORKING & CONFIGURATION

## Module 11 — Docker storage

### Storage types

| Type | Description | Common use |
|---|---|---|
| Writable container layer | Container-specific runtime writes | Temporary data |
| Volume | Docker-managed persistent storage | Database data, persistent app data |
| Bind mount | Maps a host path into container | Local development, host-managed files |
| `tmpfs` mount | Temporary in-memory filesystem on supported Linux hosts | Temporary sensitive/runtime data |

### Key concept

Removing a container removes its writable layer, but named volumes normally persist independently.

### Volume commands

```bash
docker volume create ml-data
docker volume ls
docker volume inspect ml-data
docker volume rm ml-data
docker volume prune
```

Mount named volume:

```bash
docker run -it \
  --mount type=volume,source=ml-data,target=/data \
  ubuntu bash
```

Bind mount:

```bash
docker run -it \
  --mount type=bind,source="$(pwd)",target=/app \
  ubuntu bash
```

Read-only bind mount:

```bash
docker run -it \
  --mount type=bind,source="$(pwd)/models",target=/models,readonly \
  python:3.12-slim bash
```

### Volume vs bind mount

- Volume: Docker manages the storage location.
- Bind mount: user specifies a host file or directory.
- Use volumes for Docker-managed persistent application data.
- Use bind mounts often for development and direct host-file access.

**Warning:** `docker volume prune` and `docker compose down -v` can delete persistent data.

## Module 12 — Docker networking

Networking allows containers to communicate with other containers, the host and external systems.

### Common network drivers

| Driver | Main use |
|---|---|
| `bridge` | Containers on one Docker host |
| `host` | Shares host network namespace (platform-dependent behavior) |
| `none` | No normal network connectivity |
| `overlay` | Multi-host Docker networking, e.g. Swarm |
| `macvlan` | Gives container a MAC address on a physical network |
| `ipvlan` | Network connectivity using an IP VLAN-style model |

### Default bridge vs user-defined bridge

A user-defined bridge provides automatic container-name DNS resolution and better control for application networks.

```bash
docker network ls
docker network create ai-network
docker network inspect ai-network
docker network connect ai-network backend
docker network disconnect ai-network backend
docker network rm ai-network
```

Connect containers:

```bash
docker run -d --name backend --network ai-network nginx
docker run -it --name client --network ai-network alpine sh
```

On the same user-defined network, the client can resolve the backend by name:

```text
http://backend
```

Within a container, `localhost` refers to that same container, not another service.

### AI application network example

```text
backend → database:5432
backend → model-service:8000
backend → redis:6379
```

Do not publish internal database/model ports to the host unless required.

## Module 13 — Environment variables & configuration

Environment variables configure an application without hardcoding values.

Python:

```python
import os

port = int(os.getenv("PORT", "8000"))
environment = os.getenv("APP_ENV", "development")
```

Dockerfile:

```dockerfile
ENV APP_ENV=production
ENV PORT=8000
```

Runtime override:

```bash
docker run -e APP_ENV=development -e PORT=8000 my-image
```

Environment file:

```env
APP_ENV=production
PORT=8000
MODEL_PATH=/models/model.pkl
```

```bash
docker run --env-file app.env my-image
```

### `ENV` vs `-e` vs `--env-file`

| Method | Defined where | Purpose |
|---|---|---|
| Dockerfile `ENV` | Image | Default environment |
| `-e` / `--env` | Runtime command | Set/override individual variable |
| `--env-file` | External file | Load multiple variables |

**Security:** environment variables and `.env` files are not encrypted secret stores. Do not bake credentials into images or commit real `.env` files. Use appropriate secret-management mechanisms.

### Phase 4 key takeaway

- Volumes preserve data.
- Networks connect services.
- Environment variables configure services.

# 🧩 PHASE 5 — DOCKER COMPOSE & MULTI-CONTAINER APPLICATIONS

## Module 14 — Docker Compose introduction

**Docker Compose** defines and manages multi-container applications using a YAML file, commonly `compose.yaml`.

Instead of manually running each service, define the application stack once:

```yaml
services:
  web:
    image: nginx
    ports:
      - "8080:80"

  redis:
    image: redis:alpine
```

Run:

```bash
docker compose up
docker compose up -d
```

Compose creates/uses the required containers and application network.

## Module 15 — Compose YAML & service architecture

### Compose file structure

```yaml
services:
  api:
    build: .
    ports:
      - "8000:8000"
    environment:
      APP_ENV: production
    depends_on:
      - redis

  redis:
    image: redis:alpine

volumes:
  app-data:

networks:
  app-network:
```

### Common service properties

| Property | Purpose |
|---|---|
| `image` | Use an existing image |
| `build` | Build from a Dockerfile/context |
| `ports` | Publish host-to-container ports |
| `expose` | Document/expose container port to connected services without publishing to host |
| `environment` | Set environment variables |
| `env_file` | Load environment variables from file |
| `volumes` | Mount persistent storage or host paths |
| `networks` | Connect services to networks |
| `depends_on` | Define dependency/startup relationships |
| `command` | Override default command |
| `restart` | Configure restart behavior |
| `healthcheck` | Test service health |

### Networking

Compose creates a default network for the application. Services can reach each other using service names:

```text
API → redis:6379
API → db:5432
```

### `depends_on`

Basic `depends_on` controls startup order, but does not by itself mean the dependency is ready to accept requests.

For readiness conditions, configure a health check and use:

```yaml
depends_on:
  redis:
    condition: service_healthy
```

### Health check example

```yaml
redis:
  image: redis:alpine
  healthcheck:
    test: ["CMD", "redis-cli", "ping"]
    interval: 5s
    timeout: 3s
    retries: 5
```

### Restart policies

- `no`
- `always`
- `on-failure`
- `unless-stopped`

## Module 16 — Compose operations

```bash
docker compose up
docker compose up -d
docker compose up --build -d

docker compose ps
docker compose logs
docker compose logs -f api
docker compose exec api sh
docker compose run api python --version

docker compose stop
docker compose start
docker compose restart
docker compose build
docker compose config
docker compose down
docker compose down -v
```

### `exec` vs `run`

| `exec` | `run` |
|---|---|
| Uses an existing running service container | Creates a one-off container for the service |
| Good for debugging | Good for one-off tasks, tests and migrations |

### `down` vs `down -v`

- `docker compose down`: removes Compose containers and default network.
- `docker compose down -v`: additionally removes declared named and attached anonymous volumes. Use carefully.

### Scaling

```bash
docker compose up -d --scale backend=3
```

Scaling limitations:
- Fixed host port mappings can conflict between replicas.
- Replicas need suitable shared-state handling.
- Model memory/GPU capacity and database connection limits matter.
- Compose scaling alone does not create a multi-host cluster or automatically configure a complete external load balancer.

### Phase 5 key takeaway

Compose describes and manages an application stack. YAML defines services, dependencies, ports, volumes, networks and configuration.

# 🚀 PHASE 6 — PRODUCTION DOCKER & DEPLOYMENT

## Module 17 — Logs & debugging

### First-line debugging commands

```bash
docker ps
docker ps -a
docker logs <container>
docker logs -f <container>
docker logs --tail 100 <container>
docker logs -t <container>

docker inspect <container>
docker stats
docker exec -it <container> sh
```

- `STDOUT`: normal process output.
- `STDERR`: error output.
- `docker logs`: displays logs captured by the configured logging driver.
- `docker inspect`: detailed configuration/state in JSON.
- `docker stats`: live CPU, memory, network I/O, block I/O and process metrics.

### Common exit codes

| Code | General meaning |
|---:|---|
| `0` | Success |
| `1` | General application error |
| `125` | Docker failed to run container |
| `126` | Command could not be invoked |
| `127` | Command not found |
| `137` | SIGKILL; possible OOM, not proof by itself |
| `143` | SIGTERM termination |

### Logging drivers

Examples: `json-file`, `local`, `syslog`, `journald`, `fluentd`, `awslogs`, `gcplogs`.

Example log rotation options:

```bash
docker run -d \
  --name web \
  --log-driver=json-file \
  --log-opt max-size=10m \
  --log-opt max-file=3 \
  nginx
```

### Debugging sequence

```text
1. docker ps -a
2. docker logs <container>
3. docker inspect <container>
4. Check exit code / OOMKilled / health
5. docker stats (if running)
6. Inspect application/config/dependencies/network
7. Fix → rebuild → retest
```

If a container exits too quickly for `exec`, start a temporary shell from the same image:

```bash
docker run --rm -it --entrypoint sh my-image
```

## Module 18 — Registries

A **registry** stores and distributes images.

Image reference pattern:

```text
REGISTRY / NAMESPACE / REPOSITORY : TAG
```

Example:

```text
docker.io/yuva19/sentiment-api:v1
```

### Push workflow

```bash
docker build -t sentiment-api:v1 .
docker login

docker tag sentiment-api:v1 \
  YOUR_USERNAME/sentiment-api:v1

docker push YOUR_USERNAME/sentiment-api:v1
```

### Pull and run elsewhere

```bash
docker pull YOUR_USERNAME/sentiment-api:v1

docker run -d \
  --name sentiment-api \
  -p 8000:8000 \
  YOUR_USERNAME/sentiment-api:v1
```

Other registries:
- Docker Hub
- GitHub Container Registry (`ghcr.io`)
- Amazon ECR
- Azure Container Registry
- Google Artifact Registry
- Harbor

### Tags vs digests

- Tag: convenient mutable name such as `v1` or `latest`.
- Digest: content-addressed identifier for exact image content.
- Pin versions/digests where reproducibility is important; do not blindly depend on mutable tags.

## Module 19 — Deployment

### Typical deployment workflow

```text
Code → Dockerfile → Build → Test
     → Push image to registry
     → Server pulls image
     → Run container / Compose stack
     → Health checks + logs + monitoring
```

### Linux server example

```bash
docker pull YOUR_USERNAME/sentiment-api:v1

docker run -d \
  --name sentiment-api \
  --restart unless-stopped \
  -p 8000:8000 \
  YOUR_USERNAME/sentiment-api:v1

docker ps
docker logs sentiment-api
curl http://localhost:8000/health
```

### Production considerations

- Use non-root user.
- Use trusted, maintained base images.
- Manage secrets securely.
- Publish only necessary ports.
- Configure health checks and restart policies.
- Set suitable CPU/memory limits.
- Use persistent storage and backups for important data.
- Configure TLS/reverse proxy where appropriate.
- Monitor logs, metrics and application health.
- Test updates and rollback procedures.

## Module 20 — Scaling & load balancing

### Vertical vs horizontal scaling

| Vertical | Horizontal |
|---|---|
| Add CPU/RAM to an instance | Add more instances/replicas |
| Bigger machine/container limits | More service instances |
| Still one instance failure domain | Can improve capacity/availability if designed properly |

### Load balancing

Distributes requests among multiple backend instances.

Common algorithms:
- Round robin
- Weighted round robin
- Least connections
- IP hash
- Least response time (implementation-dependent)

### Reverse proxy

A reverse proxy receives client traffic and forwards it to backend services. It can also provide TLS termination, routing, caching and load balancing depending on configuration.

Examples: NGINX, HAProxy, Traefik.

### Compose scaling

```bash
docker compose up -d --scale backend=3
docker compose ps
```

A stable public endpoint normally requires a suitable proxy/load balancer. Avoid binding every replica to the same fixed host port.

### Docker Swarm

Docker Swarm is Docker's native cluster/orchestration technology.

Key terms:
- **Manager node:** manages cluster state and scheduling.
- **Worker node:** runs assigned service tasks.
- **Service:** desired state for an application workload.
- **Replica:** instance/task of a service.
- **Overlay network:** multi-host service network.
- **Routing mesh:** routes traffic to service tasks on the Swarm.

Commands:

```bash
docker swarm init
docker node ls

docker service create \
  --name sentiment-api \
  --replicas 3 \
  -p 8000:8000 \
  sentiment-api:v1

docker service ls
docker service ps sentiment-api
docker service scale sentiment-api=5
docker service rm sentiment-api
```

Compose is primarily application definition/management; Swarm provides multi-host cluster orchestration. Compose `--scale` does not itself create a cluster.

# 🔐 PHASE 7 — SECURITY, OPTIMIZATION & REAL-WORLD AI/ML

## Module 21 — Security & best practices

### Core security practices

1. Run as a non-root user when possible.
2. Avoid `--privileged` unless specifically required.
3. Drop unnecessary Linux capabilities.
4. Use read-only root filesystems where practical.
5. Do not hardcode secrets in Dockerfiles or source code.
6. Use trusted and maintained base images.
7. Scan images for vulnerabilities.
8. Restrict published ports and network access.
9. Apply resource limits.
10. Keep images and dependencies updated.

Non-root Dockerfile:

```dockerfile
FROM python:3.12-slim

WORKDIR /app

RUN useradd --create-home appuser

COPY --chown=appuser:appuser . .

USER appuser

CMD ["python", "main.py"]
```

Read-only runtime:

```bash
docker run -d \
  --read-only \
  --tmpfs /tmp \
  my-image
```

Drop capabilities:

```bash
docker run --cap-drop=ALL my-image
```

### Secrets

- `.env` files are not secret vaults.
- Do not commit `.env` files.
- Use cloud/CI secret stores or supported secret mechanisms.
- Docker Swarm secrets are available to Swarm services as files under `/run/secrets/`.

Example:

```bash
printf "secret-value" | docker secret create db_password -
```

## Module 22 — Image optimization

### Image layers and caching

Each Dockerfile instruction can contribute a layer or build step. Cache reuse depends on instructions and their inputs.

Prefer:

```dockerfile
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt
COPY . .
```

over copying the entire project before installing dependencies.

### `.dockerignore`

```gitignore
.git
.venv
venv
__pycache__
*.pyc
.env
.ipynb_checkpoints
data/raw/
logs/
```

Exclude unnecessary files, but do not exclude files required by the build/runtime.

### Multi-stage builds

Use separate builder and runtime stages. Copy only required artifacts into the final runtime image.

```dockerfile
FROM python:3.12 AS builder
WORKDIR /app
COPY requirements.txt .
RUN pip install --prefix=/install --no-cache-dir -r requirements.txt

FROM python:3.12-slim AS runtime
WORKDIR /app
COPY --from=builder /install /usr/local
COPY . .
CMD ["python", "main.py"]
```

### BuildKit cache mount

```dockerfile
# syntax=docker/dockerfile:1
FROM python:3.12-slim
WORKDIR /app
COPY requirements.txt .
RUN --mount=type=cache,target=/root/.cache/pip \
    pip install -r requirements.txt
```

### Build secrets

Use BuildKit secret mounts for credentials required during builds; do not store credentials in `ARG`, `ENV`, image layers or source control.

### Image commands

```bash
docker images
docker history my-image
docker image inspect my-image
docker image rm my-image
docker image prune
docker image prune -a
```

## Module 23 — AI/ML deployment project revision

### Project architecture

```text
Client
  │
  ▼
FastAPI container
  ├── Loads Scikit-learn model artifact
  ├── Checks Redis cache
  └── Runs inference if cache miss
          │
          ▼
       Redis container
```

### Project components

| Component | Responsibility |
|---|---|
| Training script | Trains and saves model |
| `sentiment.joblib` | Serialized ML pipeline |
| FastAPI | Exposes prediction/health endpoints |
| Dockerfile | Packages app and dependencies |
| Compose | Runs API and Redis services |
| Redis | Caches predictions |

### Typical folder structure

```text
docker-sentiment-project/
├── app/
│   └── main.py
├── models/
│   └── sentiment.joblib
├── training/
│   └── train_model.py
├── requirements.txt
├── Dockerfile
├── compose.yaml
├── .dockerignore
└── .gitignore
```

### End-to-end steps

1. Install dependencies.
2. Train model and save artifact.
3. Write FastAPI application.
4. Write Dockerfile.
5. Create `.dockerignore` and `.gitignore`.
6. Write Compose file for API and Redis.
7. Build and start services.
8. Check status and logs.
9. Test health and prediction endpoints.
10. Publish image to registry.
11. Deploy to a server.
12. Add production security, monitoring, backups and TLS.

### Important commands

```bash
# Train model
python training/train_model.py

# Build and start
docker compose up -d --build

# Inspect
docker compose ps
docker compose logs -f api
docker compose logs -f redis

# Test health
curl http://localhost:8000/health

# Test prediction
curl -X POST http://localhost:8000/predict \
  -H "Content-Type: application/json" \
  -d '{"text":"I love this product"}'

# Inspect resources
docker stats

# Stop and remove containers/network
docker compose down
```

**Project caveat:** the small example dataset is educational only. It is not sufficient to claim real-world model quality. A real project needs representative data, proper evaluation, input validation, security, monitoring and deployment testing.

# ⚡ MASTER COMMAND CHEAT SHEET

## Images

```bash
docker images
docker pull IMAGE
docker build -t NAME:TAG .
docker tag SOURCE TARGET
docker push IMAGE
docker image inspect IMAGE
docker history IMAGE
docker image rm IMAGE
```

## Containers

```bash
docker run -d --name NAME IMAGE
docker run -it IMAGE sh
docker ps
docker ps -a
docker start NAME
docker stop NAME
docker restart NAME
docker rm NAME
docker rm -f NAME
docker exec -it NAME sh
docker logs -f NAME
docker inspect NAME
docker stats
```

## Ports

```bash
docker run -p HOST_PORT:CONTAINER_PORT IMAGE
docker run -p 127.0.0.1:8080:8000 IMAGE
```

## Volumes

```bash
docker volume create NAME
docker volume ls
docker volume inspect NAME
docker volume rm NAME
docker volume prune
```

## Networks

```bash
docker network ls
docker network create NAME
docker network inspect NAME
docker network connect NETWORK CONTAINER
docker network disconnect NETWORK CONTAINER
docker network rm NAME
```

## Compose

```bash
docker compose up -d --build
docker compose ps
docker compose logs -f
docker compose exec SERVICE sh
docker compose run SERVICE COMMAND
docker compose build
docker compose config
docker compose restart
docker compose stop
docker compose start
docker compose down
docker compose down -v
docker compose up -d --scale SERVICE=3
```

## Swarm

```bash
docker swarm init
docker node ls
docker service create --name NAME --replicas 3 IMAGE
docker service ls
docker service ps NAME
docker service scale NAME=5
docker service rm NAME
```

# 🔍 HIGH-VALUE COMPARISON TABLES

| Compare | Remember |
|---|---|
| Image vs container | Template vs instance |
| Container vs VM | Isolated process vs full guest OS |
| `RUN` vs `CMD` | Build-time command vs default runtime command |
| `CMD` vs `ENTRYPOINT` | Default command/arguments vs main executable configuration |
| `EXPOSE` vs `-p` | Documentation metadata vs actual port publishing |
| Host port vs container port | Host endpoint vs application listening port |
| Volume vs bind mount | Docker-managed path vs specified host path |
| `docker run` vs `docker exec` | Create/start a container vs run command in existing container |
| `docker create` vs `docker start` | Create stopped container vs start existing one |
| Default vs user-defined bridge | User-defined bridge supports automatic name resolution |
| `ENV` vs `-e` | Image default vs runtime injection/override |
| `depends_on` vs health check | Dependency ordering vs readiness/health test |
| Compose vs Swarm | Multi-container app management vs cluster orchestration |
| `exec` vs `run` in Compose | Existing service container vs one-off service container |
| Tag vs digest | Mutable human-friendly reference vs content identifier |
| Vertical vs horizontal scaling | More resources per instance vs more instances |
| `stop` vs `down` | Stop containers vs remove containers/network |
| `down` vs `down -v` | Remove stack resources vs also remove associated volumes |
| Build cache vs cache mount | Reuse build steps vs persist package-manager cache data |
| `.gitignore` vs `.dockerignore` | Exclude from Git vs exclude from Docker build context |

# 🧪 FINAL SELF-TEST — CHECK YOUR UNDERSTANDING

Try answering without looking back.

1. What problem does Docker solve that a Python virtual environment does not fully solve?
2. Explain the difference between a Docker image and a container.
3. What are Docker CLI, daemon and API?
4. What happens when you execute `docker run`?
5. Explain `FROM`, `WORKDIR`, `COPY`, `RUN`, `ENV`, `EXPOSE` and `CMD`.
6. Why should `requirements.txt` usually be copied before the rest of the source code?
7. What is the difference between `docker run`, `create`, `start`, `stop` and `exec`?
8. Explain `-p 8080:8000`.
9. Why should an API inside a container commonly bind to `0.0.0.0`?
10. Why does container data disappear when its writable layer is removed?
11. Compare volumes and bind mounts.
12. How do two containers communicate by service/container name?
13. Why does `localhost` inside one container not mean another container?
14. Compare Dockerfile `ENV`, runtime `-e` and `--env-file`.
15. What does Docker Compose automate?
16. What is the difference between `depends_on` and a health check?
17. Compare Compose `exec` and `run`.
18. What does `docker compose down -v` remove?
19. How do you debug a container that exits immediately?
20. What is the difference between a registry, repository, tag and digest?
21. Describe the workflow for pushing an image to Docker Hub and running it on another machine.
22. What is a restart policy? Does it fix application bugs?
23. Compare vertical and horizontal scaling.
24. What does a load balancer do?
25. What is Docker Swarm?
26. Why should containers avoid running as root unnecessarily?
27. Why are `.env` files not a secure secret-management system?
28. What is a multi-stage build?
29. What does `.dockerignore` do?
30. Describe the complete AI/ML Docker deployment workflow.

## Final recall challenge

Without notes, write the commands to:

- Build an image.
- Run it with a published port.
- View logs.
- Enter the container.
- Create and mount a volume.
- Create a custom network.
- Run a Compose stack.
- Push an image to Docker Hub.
- Inspect resource usage.
- Stop and remove the Compose stack.

---

# 🏁 FINAL MENTAL MODEL

```text
UNDERSTAND THE APPLICATION
          ↓
IDENTIFY RUNTIME & DEPENDENCIES
          ↓
WRITE DOCKERFILE
          ↓
BUILD IMAGE
          ↓
RUN CONTAINER
          ↓
CONFIGURE PORTS / STORAGE / NETWORK
          ↓
DEFINE MULTI-SERVICE STACK WITH COMPOSE
          ↓
DEBUG & MONITOR
          ↓
PUSH IMAGE TO REGISTRY
          ↓
DEPLOY TO SERVER
          ↓
SECURE / OPTIMIZE / SCALE
```

**Docker's core idea:** package an application and its dependencies into an image, run it consistently as a container, connect it to the infrastructure it needs, and manage its lifecycle reliably.